In [1]:
!pip install --upgrade protobuf

  Using cached protobuf-7.35.0-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.35.0-cp310-abi3-manylinux2014_x86_64.whl (327 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hopsworks 4.7.5 requires protobuf<5.0.0,>=4.25.4, but you have protobuf 7.35.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
google-cloud-aiplatform 1.153.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>

In [1]:
# Install Hopsworks (since this is a fresh notebook)
!pip install "hopsworks[python]==4.7.*"

import hopsworks
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Connect to Hopsworks
print("Connecting to Hopsworks...")
project = hopsworks.login() # Paste your API key here when prompted!
fs = project.get_feature_store()

# 2. Fetch your ultimate dataset
print("Downloading Version 2 data from the cloud...")
fg = fs.get_feature_group("karachi_aqi_features", version=2)
df = fg.read()

# 3. Prepare Data for 24-Hour Forecasting
print("Prepping target variables...")
df_model = df.sort_values('time').copy()
df_model['target_pm2_5_next_24h'] = df_model['pm2_5'].shift(-24)
df_model = df_model.dropna()

# Isolate features (X) and target (y)
X = df_model.drop(columns=['time', 'target_pm2_5_next_24h'])
y = df_model['target_pm2_5_next_24h']

# 4. The "Vaulted" Test Split
# We lock away the last 20% of the timeline. The tournament will NOT see this.
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("✅ Setup complete! You are ready to run the tournament.")

  Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl (295 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.0
    Uninstalling protobuf-7.35.0:
      Successfully uninstalled protobuf-7.35.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.


Connecting to Hopsworks...

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/33022
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (53.22s) 
Prepping target variables...
✅ Setup complete! You are ready to run the tournament.


In [3]:
# import pandas as pd
# import numpy as np
# import warnings
# from sklearn.model_selection import TimeSeriesSplit
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import Ridge
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.neural_network import MLPRegressor
# import xgboost as xgb
# import lightgbm as lgb
# from sklearn.metrics import mean_squared_error, r2_score

# warnings.filterwarnings('ignore')

# models = {
#     "Ridge Regression": Ridge(),
#     "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
#     "XGBoost": xgb.XGBRegressor(random_state=42),
#     "LightGBM": lgb.LGBMRegressor(random_state=42, verbose=-1),
#     "Neural Network (MLP)": MLPRegressor(random_state=42, max_iter=500)
# }

# tscv = TimeSeriesSplit(n_splits=10)
# print("Starting the Scaled Model Tournament (10-Fold Time-Series CV)...\n")

# results_summary = []

# for name, model in models.items():
#     mse_scores = []
#     rmse_scores = []
#     r2_scores = []

#     for train_idx, val_idx in tscv.split(X_train_val):
#         X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
#         X_v, y_v = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

#         # --- THE NEW MAGIC: Scaling the data inside the fold ---
#         scaler = StandardScaler()
#         X_tr_scaled = scaler.fit_transform(X_tr)
#         X_v_scaled = scaler.transform(X_v) # Transform validation set using training rules

#         # Train on scaled data
#         model.fit(X_tr_scaled, y_tr)
#         preds = model.predict(X_v_scaled)

#         mse = mean_squared_error(y_v, preds)
#         rmse = np.sqrt(mse)
#         r2 = r2_score(y_v, preds)

#         mse_scores.append(mse)
#         rmse_scores.append(rmse)
#         r2_scores.append(r2)

#     avg_mse = np.mean(mse_scores)
#     avg_rmse = np.mean(rmse_scores)
#     avg_r2 = np.mean(r2_scores)

#     print(f"--- {name} ---")
#     print(f"Avg MSE:  {avg_mse:.2f}")
#     print(f"Avg RMSE: {avg_rmse:.2f}")
#     print(f"Avg R²:   {avg_r2:.2f}\n")

#     results_summary.append({
#         "Model": name,
#         "MSE": avg_mse,
#         "RMSE": avg_rmse,
#         "R-Squared": avg_r2
#     })

# print("\n🏆 FINAL SCALED TOURNAMENT LEADERBOARD 🏆")
# leaderboard = pd.DataFrame(results_summary).sort_values(by="R-Squared", ascending=False).reset_index(drop=True)
# display(leaderboard)

Starting the Scaled Model Tournament (10-Fold Time-Series CV)...

--- Ridge Regression ---
Avg MSE:  331.87
Avg RMSE: 15.10
Avg R²:   -0.06

--- Random Forest ---
Avg MSE:  501.96
Avg RMSE: 19.21
Avg R²:   -0.99

--- XGBoost ---
Avg MSE:  473.83
Avg RMSE: 19.44
Avg R²:   -0.87

--- LightGBM ---
Avg MSE:  324.49
Avg RMSE: 15.90
Avg R²:   -0.21

--- Neural Network (MLP) ---
Avg MSE:  448.47
Avg RMSE: 18.43
Avg R²:   -1.52


🏆 FINAL SCALED TOURNAMENT LEADERBOARD 🏆


,Model,MSE,RMSE,R-Squared
0,Ridge Regression,331.867354,15.095665,-0.056559
1,LightGBM,324.490008,15.904152,-0.207056
2,XGBoost,473.825164,19.440453,-0.872768
3,Random Forest,501.961854,19.205410,-0.994258
4,Neural Network (MLP),448.473258,18.433373,-1.523850


In [3]:
import pandas as pd
import numpy as np
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# 1. Initialize our standard Competitors
models = {
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(random_state=42),
    "LightGBM": lgb.LGBMRegressor(random_state=42, verbose=-1),
    "PyTorch (Deep Learning)": "PyTorch_Placeholder" # Handled dynamically
}

tscv = TimeSeriesSplit(n_splits=10)
print("Starting the Scaled Tournament with PyTorch (10-Fold CV)...\n")

results_summary = []

for name, model in models.items():
    mse_scores = []
    rmse_scores = []
    r2_scores = []

    print(f"Training {name}...")

    for train_idx, val_idx in tscv.split(X_train_val):
        X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
        X_v, y_v = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

        # Scale the data inside the fold
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)

        # --- PYTORCH LOGIC ---
        if name == "PyTorch (Deep Learning)":
            # Convert numpy arrays to PyTorch Tensors
            X_train_t = torch.tensor(X_tr_scaled, dtype=torch.float32)
            y_train_t = torch.tensor(y_tr.values, dtype=torch.float32).view(-1, 1)
            X_val_t = torch.tensor(X_v_scaled, dtype=torch.float32)

            # Build the Neural Network Architecture
            pt_model = nn.Sequential(
                nn.Linear(X_tr_scaled.shape[1], 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

            # Loss and Optimizer
            criterion = nn.MSELoss()
            optimizer = optim.Adam(pt_model.parameters(), lr=0.005)

            # Train for 30 Epochs
            for epoch in range(30):
                optimizer.zero_grad()
                outputs = pt_model(X_train_t)
                loss = criterion(outputs, y_train_t)
                loss.backward()
                optimizer.step()

            # Predict
            with torch.no_grad():
                preds = pt_model(X_val_t).numpy().flatten()

        # --- STANDARD ML LOGIC ---
        else:
            model.fit(X_tr_scaled, y_tr)
            preds = model.predict(X_v_scaled)

        # Calculate metrics
        mse = mean_squared_error(y_v, preds)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_v, preds)

        mse_scores.append(mse)
        rmse_scores.append(rmse)
        r2_scores.append(r2)

    avg_mse = np.mean(mse_scores)
    avg_rmse = np.mean(rmse_scores)
    avg_r2 = np.mean(r2_scores)

    # Save to our results list
    results_summary.append({
        "Model": name,
        "MSE": avg_mse,
        "RMSE": avg_rmse,
        "R-Squared": avg_r2
    })

# 4. Display the Final Leaderboard (Ranked by RMSE)
print("\n🏆 FINAL SCALED TOURNAMENT LEADERBOARD (Ranked by RMSE) 🏆")
leaderboard = pd.DataFrame(results_summary).sort_values(by="RMSE", ascending=True).reset_index(drop=True)
display(leaderboard)

Starting the Scaled Tournament with PyTorch (10-Fold CV)...

Training Ridge Regression...
Training Random Forest...
Training XGBoost...
Training LightGBM...
Training PyTorch (Deep Learning)...

🏆 FINAL SCALED TOURNAMENT LEADERBOARD (Ranked by RMSE) 🏆


,Model,MSE,RMSE,R-Squared
0,Ridge Regression,331.867354,15.095665,-0.056559
1,LightGBM,324.490008,15.904152,-0.207056
2,PyTorch (Deep Learning),394.231449,18.401084,-0.827945
3,Random Forest,501.961854,19.205410,-0.994258
4,XGBoost,473.825164,19.440453,-0.872768


In [4]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

tscv = TimeSeriesSplit(n_splits=5) # 5 folds for faster tuning

print("--- Round 1: Wide Tuning for Ridge Regression ---")
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

# WIDE RANGE: From 0.001 to 10,000 using logspace
ridge_param_dist = {
    'model__alpha': np.logspace(-3, 4, 100)
}

ridge_search = RandomizedSearchCV(
    ridge_pipeline, param_distributions=ridge_param_dist,
    n_iter=30, cv=tscv, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1
)
ridge_search.fit(X_train_val, y_train_val)

print(f"✅ Best Ridge Params: {ridge_search.best_params_}")
print(f"🏆 Best Ridge RMSE:  {-ridge_search.best_score_:.2f}")

--- Round 1: Wide Tuning for Ridge Regression ---
✅ Best Ridge Params: {'model__alpha': np.float64(739.0722033525775)}
🏆 Best Ridge RMSE:  12.47


In [5]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

tscv = TimeSeriesSplit(n_splits=5)

print("--- Round 1: Wide Tuning for LightGBM ---")
lgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', lgb.LGBMRegressor(random_state=42, verbose=-1))
])

# WIDE RANGE: Testing massive variance in trees, depth, and learning rates
lgb_param_dist = {
    'model__n_estimators': [50, 100, 200, 500, 1000],
    'model__learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2],
    'model__max_depth': [3, 5, 7, 10, -1], # -1 means no limit
    'model__num_leaves': [15, 31, 63, 127],
    'model__subsample': [0.5, 0.7, 0.9, 1.0],
    'model__colsample_bytree': [0.5, 0.7, 0.9, 1.0]
}

lgb_search = RandomizedSearchCV(
    lgb_pipeline, param_distributions=lgb_param_dist,
    n_iter=30, cv=tscv, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1
)
lgb_search.fit(X_train_val, y_train_val)

print(f"✅ Best LightGBM Params: {lgb_search.best_params_}")
print(f"🏆 Best LightGBM RMSE:  {-lgb_search.best_score_:.2f}")

--- Round 1: Wide Tuning for LightGBM ---
✅ Best LightGBM Params: {'model__subsample': 0.7, 'model__num_leaves': 15, 'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.7}
🏆 Best LightGBM RMSE:  12.35


In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

tscv = TimeSeriesSplit(n_splits=5)

print("--- Round 1: Wide Tuning for Random Forest ---")
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# WIDE RANGE: Testing small constrained trees vs massive deep forests
rf_param_dist = {
    'model__n_estimators': [50, 100, 200, 500],
    'model__max_depth': [3, 5, 10, 20, None],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 5, 10]
}

rf_search = RandomizedSearchCV(
    rf_pipeline, param_distributions=rf_param_dist,
    n_iter=20, cv=tscv, scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1
)
rf_search.fit(X_train_val, y_train_val)

print(f"✅ Best Random Forest Params: {rf_search.best_params_}")
print(f"🏆 Best Random Forest RMSE:  {-rf_search.best_score_:.2f}")

--- Round 1: Wide Tuning for Random Forest ---
✅ Best Random Forest Params: {'model__n_estimators': 500, 'model__min_samples_split': 20, 'model__min_samples_leaf': 10, 'model__max_depth': 3}
🏆 Best Random Forest RMSE:  12.48


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import numpy as np

tscv = TimeSeriesSplit(n_splits=5)
print("--- Round 1: Wide Tuning for PyTorch ---")

# WIDE RANGE: Testing different brain sizes and learning speeds
pt_param_space = {
    'lr': [0.001, 0.005, 0.01, 0.05],
    'hidden_1': [32, 64, 128, 256],
    'hidden_2': [16, 32, 64, 128],
    'epochs': [20, 50, 100]
}

n_iter = 15
best_pt_rmse = float('inf')
best_pt_params = {}

for i in range(n_iter):
    # Pick a random combination of parameters
    config = {k: random.choice(v) for k, v in pt_param_space.items()}
    fold_rmses = []

    for train_idx, val_idx in tscv.split(X_train_val):
        X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
        X_v, y_v = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

        # Safe Scaling
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)

        X_train_t = torch.tensor(X_tr_scaled, dtype=torch.float32)
        y_train_t = torch.tensor(y_tr.values, dtype=torch.float32).view(-1, 1)
        X_val_t = torch.tensor(X_v_scaled, dtype=torch.float32)

        # Build the dynamic model
        model = nn.Sequential(
            nn.Linear(X_tr_scaled.shape[1], config['hidden_1']),
            nn.ReLU(),
            nn.Linear(config['hidden_1'], config['hidden_2']),
            nn.ReLU(),
            nn.Linear(config['hidden_2'], 1)
        )

        optimizer = optim.Adam(model.parameters(), lr=config['lr'])
        criterion = nn.MSELoss()

        # Train silently
        for epoch in range(config['epochs']):
            optimizer.zero_grad()
            loss = criterion(model(X_train_t), y_train_t)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            preds = model(X_val_t).numpy().flatten()
            fold_rmses.append(np.sqrt(mean_squared_error(y_v, preds)))

    avg_rmse = np.mean(fold_rmses)
    if avg_rmse < best_pt_rmse:
        best_pt_rmse = avg_rmse
        best_pt_params = config

print(f"✅ Best PyTorch Params: {best_pt_params}")
print(f"🏆 Best PyTorch RMSE:  {best_pt_rmse:.2f}")

--- Round 1: Wide Tuning for PyTorch ---
✅ Best PyTorch Params: {'lr': 0.05, 'hidden_1': 32, 'hidden_2': 64, 'epochs': 50}
🏆 Best PyTorch RMSE:  14.81


In [8]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

tscv = TimeSeriesSplit(n_splits=5)

print("--- Round 2: Narrow Tuning for Ridge Regression ---")
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

# NARROW RANGE: Surrounding ~739
ridge_param_grid = {
    'model__alpha': [600.0, 650.0, 700.0, 739.0, 750.0, 800.0, 850.0, 900.0]
}

ridge_grid = GridSearchCV(
    ridge_pipeline, param_grid=ridge_param_grid,
    cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1
)
ridge_grid.fit(X_train_val, y_train_val)

best_ridge = ridge_grid.best_estimator_
print(f"✅ Final Ridge Params: {ridge_grid.best_params_}")
print(f"🏆 Final Ridge RMSE:  {-ridge_grid.best_score_:.2f}")

--- Round 2: Narrow Tuning for Ridge Regression ---
✅ Final Ridge Params: {'model__alpha': 900.0}
🏆 Final Ridge RMSE:  12.47


In [9]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

tscv = TimeSeriesSplit(n_splits=5)

print("--- Round 2: Narrow Tuning for LightGBM ---")
lgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', lgb.LGBMRegressor(random_state=42, verbose=-1))
])

# NARROW RANGE: Tightly packed around the Round 1 winners
lgb_param_grid = {
    'model__n_estimators': [180, 200, 220],
    'model__learning_rate': [0.008, 0.01, 0.015],
    'model__max_depth': [2, 3, 4],
    'model__num_leaves': [12, 15, 18],
    'model__subsample': [0.65, 0.7, 0.75],
    'model__colsample_bytree': [0.65, 0.7, 0.75]
}

lgb_grid = GridSearchCV(
    lgb_pipeline, param_grid=lgb_param_grid,
    cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1
)
lgb_grid.fit(X_train_val, y_train_val)

best_lgb = lgb_grid.best_estimator_
print(f"✅ Final LightGBM Params: {lgb_grid.best_params_}")
print(f"🏆 Final LightGBM RMSE:  {-lgb_grid.best_score_:.2f}")

--- Round 2: Narrow Tuning for LightGBM ---
✅ Final LightGBM Params: {'model__colsample_bytree': 0.75, 'model__learning_rate': 0.015, 'model__max_depth': 2, 'model__n_estimators': 220, 'model__num_leaves': 12, 'model__subsample': 0.65}
🏆 Final LightGBM RMSE:  12.09


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

tscv = TimeSeriesSplit(n_splits=5)

print("--- Round 2: Narrow Tuning for Random Forest ---")
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# NARROW RANGE: Tightly packed around the Round 1 winners
rf_param_grid = {
    'model__n_estimators': [450, 500, 550],
    'model__max_depth': [2, 3, 4],
    'model__min_samples_split': [18, 20, 22],
    'model__min_samples_leaf': [8, 10, 12]
}

rf_grid = GridSearchCV(
    rf_pipeline, param_grid=rf_param_grid,
    cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1
)
rf_grid.fit(X_train_val, y_train_val)

best_rf = rf_grid.best_estimator_
print(f"✅ Final Random Forest Params: {rf_grid.best_params_}")
print(f"🏆 Final Random Forest RMSE:  {-rf_grid.best_score_:.2f}")

--- Round 2: Narrow Tuning for Random Forest ---
✅ Final Random Forest Params: {'model__max_depth': 3, 'model__min_samples_leaf': 12, 'model__min_samples_split': 18, 'model__n_estimators': 500}
🏆 Final Random Forest RMSE:  12.37


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

tscv = TimeSeriesSplit(n_splits=5)
print("--- Round 2: Narrow Tuning for PyTorch ---")

# NARROW RANGE: A micro-grid to squeeze out the best architecture
pt_param_grid = [
    {'lr': lr, 'hidden_1': h1, 'hidden_2': h2, 'epochs': ep}
    for lr in [0.04, 0.05, 0.06]
    for h1 in [28, 32, 36]
    for h2 in [56, 64, 72]
    for ep in [45, 50, 55]
]

best_pt_rmse = float('inf')
best_pt_params = {}
best_pt_model_state = None

for config in pt_param_grid:
    fold_rmses = []

    for train_idx, val_idx in tscv.split(X_train_val):
        X_tr, y_tr = X_train_val.iloc[train_idx], y_train_val.iloc[train_idx]
        X_v, y_v = X_train_val.iloc[val_idx], y_train_val.iloc[val_idx]

        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)

        X_train_t = torch.tensor(X_tr_scaled, dtype=torch.float32)
        y_train_t = torch.tensor(y_tr.values, dtype=torch.float32).view(-1, 1)
        X_val_t = torch.tensor(X_v_scaled, dtype=torch.float32)

        model = nn.Sequential(
            nn.Linear(X_tr_scaled.shape[1], config['hidden_1']),
            nn.ReLU(),
            nn.Linear(config['hidden_1'], config['hidden_2']),
            nn.ReLU(),
            nn.Linear(config['hidden_2'], 1)
        )

        optimizer = optim.Adam(model.parameters(), lr=config['lr'])
        criterion = nn.MSELoss()

        for epoch in range(config['epochs']):
            optimizer.zero_grad()
            loss = criterion(model(X_train_t), y_train_t)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            preds = model(X_val_t).numpy().flatten()
            fold_rmses.append(np.sqrt(mean_squared_error(y_v, preds)))

    avg_rmse = np.mean(fold_rmses)
    if avg_rmse < best_pt_rmse:
        best_pt_rmse = avg_rmse
        best_pt_params = config

print(f"✅ Final PyTorch Params: {best_pt_params}")
print(f"🏆 Final PyTorch RMSE:  {best_pt_rmse:.2f}")

--- Round 2: Narrow Tuning for PyTorch ---
✅ Final PyTorch Params: {'lr': 0.06, 'hidden_1': 32, 'hidden_2': 64, 'epochs': 50}
🏆 Final PyTorch RMSE:  13.43


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
import lightgbm as lgb

print("="*60)
print("🚀 PHASE 4a: BUILDING & TRAINING ALL FINAL MODELS 🚀")
print("="*60)

# 1. Hardcoding the winners
best_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=900.0))
])

best_lgb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', lgb.LGBMRegressor(
        colsample_bytree=0.75, learning_rate=0.015, max_depth=2,
        n_estimators=220, num_leaves=12, subsample=0.65,
        random_state=42, verbose=-1
    ))
])

best_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(
        max_depth=3, min_samples_leaf=12, min_samples_split=18,
        n_estimators=500, random_state=42, n_jobs=-1
    ))
])

# 2. Building the Voting Ensemble (The Time-Series Safe Method!)
estimators = [
    ('Ridge', best_ridge),
    ('LGBM', best_lgb),
    ('RF', best_rf)
]

# VotingRegressor just securely averages the predictions of the 3 models
voting_ensemble = VotingRegressor(estimators=estimators, n_jobs=-1)

# 3. Fit EVERYTHING on the training data
print("Fitting Ridge, LightGBM, and Random Forest...")
best_ridge.fit(X_train_val, y_train_val)
best_lgb.fit(X_train_val, y_train_val)
best_rf.fit(X_train_val, y_train_val)

print("Fitting the Voting Ensemble...")
voting_ensemble.fit(X_train_val, y_train_val)

print("\n✅ All models and the Ensemble are successfully trained and ready for testing!")

🚀 PHASE 4a: BUILDING & TRAINING ALL FINAL MODELS 🚀
Fitting Ridge, LightGBM, and Random Forest...
Fitting the Voting Ensemble...

✅ All models and the Ensemble are successfully trained and ready for testing!


In [16]:
import joblib
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("="*60)
print("🏆 PHASE 4b: FINAL EVALUATION ON UNSEEN VAULTED DATA 🏆")
print("="*60)

def evaluate_vaulted_data(name, pipeline):
    preds = pipeline.predict(X_test)
    r2 = r2_score(y_test, preds)
    print(f"\n[{name}] Final Real-World Metrics:")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")
    print(f"MAE:  {mean_absolute_error(y_test, preds):.2f}")
    print(f"R²:   {r2:.2f}")
    return r2

# Run the test data through all 4 architectures
ridge_final_r2 = evaluate_vaulted_data("Ultimate Ridge", best_ridge)
lgb_final_r2 = evaluate_vaulted_data("Ultimate LightGBM", best_lgb)
rf_final_r2 = evaluate_vaulted_data("Ultimate Random Forest", best_rf)

# THE FIX: We use the time-series safe 'voting_ensemble' here!
ensemble_final_r2 = evaluate_vaulted_data("🔥 Voting Ensemble 🔥", voting_ensemble)

print("\n" + "="*60)
print("☁️ PHASE 5: PUSHING MODELS TO HOPSWORKS REGISTRY ☁️")
print("="*60)

mr = project.get_model_registry()

def save_and_push(model, name, desc, r2):
    file_name = f"{name}.pkl"
    joblib.dump(model, file_name)
    hw_model = mr.python.create_model(
        name=name,
        metrics={"Final_Test_R2": r2},
        description=desc
    )
    hw_model.save(file_name)
    print(f"✅ Successfully pushed {name} to Hopsworks!")

# We push Ridge, LGBM, and the Ensemble!
save_and_push(best_ridge, "karachi_ridge_aqi_final", "Ultimate Ridge (24h PM2.5)", ridge_final_r2)
save_and_push(best_lgb, "karachi_lgb_aqi_final", "Ultimate LightGBM (24h PM2.5)", lgb_final_r2)
save_and_push(voting_ensemble, "karachi_ensemble_aqi_final", "Voting Ensemble (Ridge+LGBM+RF)", ensemble_final_r2)

print("\n🎉 END-TO-END PIPELINE COMPLETE! 🎉")

🏆 PHASE 4b: FINAL EVALUATION ON UNSEEN VAULTED DATA 🏆

[Ultimate Ridge] Final Real-World Metrics:
RMSE: 11.15
MAE:  8.10
R²:   0.34

[Ultimate LightGBM] Final Real-World Metrics:
RMSE: 11.03
MAE:  8.06
R²:   0.35

[Ultimate Random Forest] Final Real-World Metrics:
RMSE: 11.29
MAE:  8.08
R²:   0.32

[🔥 Voting Ensemble 🔥] Final Real-World Metrics:
RMSE: 11.02
MAE:  7.98
R²:   0.35

☁️ PHASE 5: PUSHING MODELS TO HOPSWORKS REGISTRY ☁️


  0%|          | 0/6 [00:00<?, ?it/s]

Uploading /content/karachi_ridge_aqi_final.pkl: 0.000%|          | 0/2321 elapsed<00:00 remaining<?

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/33022/models/karachi_ridge_aqi_final/1
✅ Successfully pushed karachi_ridge_aqi_final to Hopsworks!


  0%|          | 0/6 [00:00<?, ?it/s]

Uploading /content/karachi_lgb_aqi_final.pkl: 0.000%|          | 0/121527 elapsed<00:00 remaining<?

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/33022/models/karachi_lgb_aqi_final/1
✅ Successfully pushed karachi_lgb_aqi_final to Hopsworks!


  0%|          | 0/6 [00:00<?, ?it/s]

Uploading /content/karachi_ensemble_aqi_final.pkl: 0.000%|          | 0/1699253 elapsed<00:00 remaining<?

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/33022/models/karachi_ensemble_aqi_final/1
✅ Successfully pushed karachi_ensemble_aqi_final to Hopsworks!

🎉 END-TO-END PIPELINE COMPLETE! 🎉
